# Hessian-SIM Reconstruction Tutorial

**Paper**: Huang et al., "Fast, long-term, super-resolution imaging with Hessian structured illumination microscopy", *Nature Biotechnology*, 2018.

**Simulation data**: Supplementary Figure 8 — synthetic 2-beam SIM with Poisson + Gaussian noise.

---

### Pipeline

1. **Background subtraction** (99 a.u.) and measured OTF loading
2. **Parameter estimation** — pattern frequency via cross-correlation
3. **Modulation/phase estimation** — EMD histogram analysis
4. **Wiener-SIM reconstruction** — phase separation, Moire frequency unmixing, Wiener filter
5. **Hessian denoising** — Split-Bregman with 2nd-order regularization ($\lambda_B = 0.5$)
6. **TV denoising** — Split-Bregman with 1st-order regularization (comparison)

In [1]:
import sys, os, time
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft2, fftshift
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import tifffile
from src.physics import generate_otf, pad_to_size, shift_otf
from src.preprocessing import (
    estimate_sim_parameters, estimate_modulation_and_phase,
    wiener_sim_reconstruct, running_average,
)
from src.solver import hessian_denoise, tv_denoise
from src.visualization import plot_comparison, plot_line_profiles, plot_hessian_vs_tv

plt.rcParams.update({'figure.figsize': (14, 8), 'figure.dpi': 100,
                     'font.size': 12, 'image.cmap': 'hot', 'image.interpolation': 'nearest'})
get_slice = lambda d: np.squeeze(d)
print('Imports OK')

ModuleNotFoundError: No module named 'tifffile'

## 1. Physical Model: Structured Illumination and the Moire Effect

### 1.1 Diffraction Limit

In conventional fluorescence microscopy, the observed image is a convolution:

$$D(\mathbf{r}) = H(\mathbf{r}) \otimes S(\mathbf{r}) + n(\mathbf{r})$$

The OTF $\tilde{H}(\mathbf{k})$ is zero beyond cutoff $k_c = 2\text{NA}/\lambda$. For this system (NA$_{\text{det}}$ = 1.49, $\lambda$ = 488 nm), the resolution limit is ~164 nm.

### 1.2 Structured Illumination and Moire Effect

SIM illuminates with sinusoidal patterns: $I_k(\mathbf{r}) = I_0[1 + m\cos(\mathbf{k}_p \cdot \mathbf{r} + \phi_k)]$

In Fourier domain, this creates three components:

$$\tilde{D}_k(\mathbf{k}) = \tilde{H}(\mathbf{k}) \left[ \tilde{S}_0(\mathbf{k}) + \frac{m}{2}e^{j\phi_k} \tilde{S}_{+1}(\mathbf{k} - \mathbf{k}_p) + \frac{m}{2}e^{-j\phi_k} \tilde{S}_{-1}(\mathbf{k} + \mathbf{k}_p) \right]$$

The $\tilde{S}_{\pm 1}$ terms contain high-frequency info shifted into the passband — this is the **Moire effect**.

### 1.3 Key distinction: Detection NA vs Excitation NA

- **Detection NA = 1.49**: determines the OTF cutoff (what the camera can resolve)
- **Excitation NA = 0.9**: determines the illumination pattern frequency $\mathbf{k}_p$ (how far the frequency bands shift)
- The filename `NA0.9` refers to excitation NA, NOT detection NA

In [2]:
# Parameters (from Supplementary Figure 8)
nangles, nphases = 2, 3
wavelength, pixel_size = 488, 65  # nm
na_det, na_exc = 1.49, 0.9
background = 99.0
regul = 2 * np.pi
spjg = [4, 4, 3]

# Load raw data and subtract background
raw_original = tifffile.imread('../data/observation/Simulation_twobeam_noise_NA0.9.tif').astype(np.float64)
print(f'Raw data: {raw_original.shape}, range [{raw_original.min():.0f}, {raw_original.max():.0f}]')

raw = raw_original - background
raw[raw < 0] = 0
print(f'After bg subtraction (bg={background}): range [{raw.min():.0f}, {raw.max():.0f}]')
print(f'2-beam SIM: {nangles} angles x {nphases} phases = {nangles*nphases} frames/timepoint')
print(f'Timepoints: {raw.shape[0] // (nangles*nphases)}')

# Show raw SIM frames
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i in range(6):
    ax = axes[i // 3, i % 3]
    ax.imshow(raw[i], cmap='gray', vmin=0, vmax=raw[:6].max())
    ax.set_title(f'Angle {i//3}, Phase {i%3}', fontsize=13, fontweight='bold')
    ax.axis('off')
plt.suptitle('Raw SIM frames (background-subtracted)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

Raw data: (72, 256, 256), range [2, 691]
After bg subtraction (bg=99.0): range [0, 592]
2-beam SIM: 2 angles x 3 phases = 6 frames/timepoint
Timepoints: 12


/var/folders/qx/lb92vzcj0ld5nrh4q5_91_q40000gn/T/ipykernel_65313/31673319.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Fourier domain: sideband peaks from structured illumination
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i in range(6):
    ax = axes[i // 3, i % 3]
    ft = np.log1p(np.abs(fftshift(fft2(raw[i]))))
    ax.imshow(ft, cmap='viridis')
    ax.set_title(f'FFT: Angle {i//3}, Phase {i%3}', fontsize=13)
    ax.axis('off')
plt.suptitle('Fourier domain: DC + sideband peaks from structured illumination', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

/var/folders/qx/lb92vzcj0ld5nrh4q5_91_q40000gn/T/ipykernel_65313/4248109877.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. OTF: Measured vs Synthetic

The MATLAB code uses a **measured OTF** from 100nm fluorescent beads, not a synthetic approximation. The measured OTF has a narrower effective support than the ideal triangle, which is critical for:
- Correct Wiener filtering (properly suppresses noise in weak OTF regions)
- Accurate pattern frequency estimation
- Visible Hessian denoising effect (synthetic OTF over-estimates the signal support)

In [4]:
# Load measured OTF and compare with synthetic
n = max(raw.shape[1], raw.shape[2], 512)
measured_otf = tifffile.imread('../data/metadata/488OTF_512.tif').astype(np.float64)
measured_otf = measured_otf / measured_otf.max()
synthetic_otf = generate_otf(n, na_det, wavelength, pixel_size)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
mid = n // 2
axes[0].plot(synthetic_otf[mid, mid:], 'b-', linewidth=2, label=f'Synthetic (NA={na_det})')
axes[0].plot(measured_otf[mid, mid:], 'r-', linewidth=2, label='Measured 488nm OTF')
axes[0].set_xlabel('Frequency (pixels)')
axes[0].set_ylabel('|OTF|')
axes[0].set_title('OTF Cross-section')
axes[0].legend()
axes[0].set_xlim(0, 250)
axes[0].grid(True, alpha=0.3)

axes[1].imshow(synthetic_otf, cmap='hot')
axes[1].set_title('Synthetic OTF (too wide)')
axes[1].axis('off')

axes[2].imshow(measured_otf, cmap='hot')
axes[2].set_title('Measured OTF (used for reconstruction)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

otf = measured_otf  # use measured OTF for reconstruction

/var/folders/qx/lb92vzcj0ld5nrh4q5_91_q40000gn/T/ipykernel_65313/2693849217.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Parameter estimation
pg = int(np.ceil(512 * 2 * na_exc * pixel_size / wavelength * (n / 512)))
fanwei = int(np.ceil(50 * (n / 512)))
n_est = nangles * nphases

print(f'Expected pattern frequency pg = {pg} px (from excitation NA={na_exc})')
zuobiaox, zuobiaoy = estimate_sim_parameters(raw[:n_est], otf, nangles, nphases, n, pg, fanwei, regul, spjg)

# Modulation depth and phase estimation
weilac, mu, sigma_z = 2.0, 150.0, 1.0
c6, angle6 = estimate_modulation_and_phase(raw[:n_est], otf, zuobiaox, zuobiaoy,
                                            nangles, nphases, n, regul, spjg, wavelength)
print(f'\nModulation depths c6 = {c6}')
print(f'Phase offsets angle6 = {angle6}')

Expected pattern frequency pg = 123 px (from excitation NA=0.9)


  Angle 0: pattern freq at (511.9707, 634.7496)


  Angle 1: pattern freq at (634.7627, 512.0198)


  Order 0: angle6=-2.7416, c6=0.5600


  Order 1: angle6=2.4184, c6=0.3600


  Order 2: angle6=-2.0816, c6=0.5000


  Order 3: angle6=-0.0016, c6=0.3000

Modulation depths c6 = [0.56 0.36 0.5  0.3 ]
Phase offsets angle6 = [-2.74159265e+00  2.41840735e+00 -2.08159265e+00 -1.59265359e-03]


## 3. Wiener-SIM Reconstruction (Moire Frequency Unmixing)

Phase separation via $M^{-1}$, then frequency shift via Fourier shift theorem:
$$\tilde{S}_{\pm 1}^{\text{shifted}}(\mathbf{k}) = \mathcal{F}\left[\mathcal{F}^{-1}[\tilde{S}_{\pm 1}] \cdot e^{\pm j\mathbf{k}_p \cdot \mathbf{r}}\right]$$

Wiener recombination: $\hat{S}(\mathbf{k}) = \frac{\sum_i w_i \tilde{S}_i \tilde{H}_i^*}{\sum_i |\tilde{H}_i|^2 + \alpha \lambda_W^2} \cdot A(\mathbf{k})$

In [6]:
t0 = time.time()
sim_result = wiener_sim_reconstruct(raw, otf, zuobiaox, zuobiaoy, c6, angle6,
                                     nangles, nphases, n, weilac, regul, spjg)
print(f'Wiener-SIM: {time.time()-t0:.1f}s, output shape {sim_result.shape} (2x super-resolved)')

# Widefield vs SIM
from scipy.ndimage import zoom as ndizoom
widefield = raw.reshape(-1, nangles*nphases, 256, 256).mean(axis=1).mean(axis=0)
wf_up = ndizoom(widefield, 2, order=3)
t = sim_result.shape[0] // 2
img_sim = get_slice(sim_result[t])
vmax = np.percentile(img_sim[img_sim > 0], 99.5)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes[0,0].imshow(wf_up, cmap='hot', vmin=0, vmax=wf_up.max()*0.8)
axes[0,0].set_title('Widefield (diffraction-limited)', fontsize=14, fontweight='bold')
axes[0,0].axis('off')
axes[0,1].imshow(img_sim, cmap='hot', vmin=0, vmax=vmax)
axes[0,1].set_title('Wiener-SIM (2x super-resolved)', fontsize=14, fontweight='bold')
axes[0,1].axis('off')

ft_wf = np.log1p(np.abs(fftshift(fft2(wf_up))))
ft_sim = np.log1p(np.abs(fftshift(fft2(img_sim))))
fmax = max(ft_wf.max(), ft_sim.max())
axes[1,0].imshow(ft_wf, cmap='viridis', vmin=0, vmax=fmax)
axes[1,0].set_title('Widefield FFT (cutoff at $k_c$)', fontsize=14, fontweight='bold')
axes[1,0].axis('off')
axes[1,1].imshow(ft_sim, cmap='viridis', vmin=0, vmax=fmax)
axes[1,1].set_title('SIM FFT (extended to $2k_c$ via Moire)', fontsize=14, fontweight='bold')
axes[1,1].axis('off')
plt.tight_layout()
plt.show()

  Modulation depth (xishu): [1.         2.28174603 2.28174603 1.         2.66666667 2.66666667]


Wiener-SIM: 1.0s, output shape (12, 512, 512) (2x super-resolved)


/var/folders/qx/lb92vzcj0ld5nrh4q5_91_q40000gn/T/ipykernel_65313/1170356039.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Hessian Denoising (the Inverse Problem)

The Wiener-SIM output $y$ has amplified noise. The Hessian denoiser solves:

$$\min_x \frac{\mu}{2}\|x - y\|_2^2 + \|D_{xx}x\|_1 + \|D_{yy}x\|_1 + \sigma^2\|D_{zz}x\|_1 + 2\|D_{xy}x\|_1 + 2\sigma\|D_{xz}x\|_1 + 2\sigma\|D_{yz}x\|_1$$

### Split-Bregman iteration (with correct carry-over structure)

```
frac = (mu/lambda) * y                     # initial
for it = 1..100:
    x = IFFT[ FFT(frac) / divide ]         # x-update (uses frac from PREVIOUS iter)
    frac = (mu/lambda) * y                  # reset to data term
    for each Hessian component i:
        d_i = shrink(D_i x + b_i, 1/lambda) # d-update (soft threshold)
        b_i += D_i x - d_i                  # b-update (Bregman variable)
        frac += alpha_i * D_i^T(d_i - b_i)  # adjoint → NEXT iter's x-update
```

**Critical**: `frac` must carry the adjoint terms across iterations. If reset at the top of the loop (before x-update), the regularization has no effect.

**Hessian vs TV**: Hessian penalizes curvature (2nd derivatives), preserving thin curvilinear structures. TV penalizes gradients (1st derivatives), producing piecewise-constant results that over-smooth fine details.

In [7]:
# Hessian denoising (lambda=0.5 from Bregman_Hessian_Denoise.m)
print('=== Hessian denoising (100 iterations, lambda=0.5) ===')
t0 = time.time()
hessian_result = hessian_denoise(sim_result, mu=mu, sigma_z=sigma_z, n_iter=100, lamda=0.5)
print(f'Done: {time.time()-t0:.1f}s')

# TV denoising (comparison)
print('\n=== TV denoising (100 iterations) ===')
t0 = time.time()
tv_result = tv_denoise(sim_result, mu=mu, n_iter=100)
print(f'Done: {time.time()-t0:.1f}s')

# Quantify denoising effect
diff_h = np.abs(sim_result - hessian_result)
diff_t = np.abs(sim_result - tv_result)
print(f'\nHessian change: {diff_h.mean()/sim_result.mean()*100:.2f}% relative, max_diff={diff_h.max():.2f}')
print(f'TV change:      {diff_t.mean()/sim_result.mean()*100:.2f}% relative, max_diff={diff_t.max():.2f}')

=== Hessian denoising (100 iterations, lambda=0.5) ===


Done: 22.1s

=== TV denoising (100 iterations) ===


Done: 18.1s

Hessian change: 5.25% relative, max_diff=9.77
TV change:      1.36% relative, max_diff=3.47


## 5. Results

In [8]:
# Full comparison (matching paper Supplementary Figure 8 layout)
plot_comparison(widefield, sim_result, hessian_result, tv_result, raw_frame=raw[0])

<Figure size 2500x1000 with 10 Axes>

In [9]:
# Zoomed comparison
img_hess = get_slice(hessian_result[t])
img_tv = get_slice(tv_result[t])
cy, cx = img_sim.shape[0]//2, img_sim.shape[1]//2
qs = min(cy, cx) // 2

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, (img, title) in enumerate(zip(
    [wf_up[cy-qs:cy+qs, cx-qs:cx+qs] / wf_up.max() * vmax,
     img_sim[cy-qs:cy+qs, cx-qs:cx+qs],
     img_hess[cy-qs:cy+qs, cx-qs:cx+qs],
     img_tv[cy-qs:cy+qs, cx-qs:cx+qs]],
    ['Widefield', 'Wiener-SIM (noisy input)', 'Hessian-SIM', 'TV-SIM'])):
    axes[i].imshow(img, cmap='hot', vmin=0, vmax=vmax)
    axes[i].set_title(title, fontsize=12, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('Zoom: Hessian preserves fine structures better than TV', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

/var/folders/qx/lb92vzcj0ld5nrh4q5_91_q40000gn/T/ipykernel_65313/942348790.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
plot_line_profiles(sim_result, hessian_result, tv_result)

<Figure size 1600x500 with 2 Axes>

In [11]:
plot_hessian_vs_tv(hessian_result, tv_result)

<Figure size 1500x500 with 3 Axes>

## 6. Summary

| Stage | Module | Key operation | Physics |
|-------|--------|--------------|---------|
| Background subtraction | `main.py` | raw -= 99 | Remove detector offset (99 a.u.) |
| Parameter estimation | `preprocessing.py` | Cross-correlation + sub-pixel | Find pattern freq $\mathbf{k}_p$ |
| Modulation estimation | `preprocessing.py` | EMD histogram smoothing | Estimate $m$, $\Delta\phi$ per angle |
| Wiener-SIM | `preprocessing.py` | $M^{-1}D \to$ freq shift $\to$ Wiener | Moire unmixing $\to$ 2x SR |
| Hessian denoise | `solver.py` | Split-Bregman ($\lambda_B=0.5$) | Remove noise, preserve curves |

### Key lessons learned during implementation

1. **Use measured OTF**, not synthetic — the measured OTF is narrower, leading to proper noise suppression
2. **Subtract background** (99 a.u.) — biases DC component and distorts frequency unmixing
3. **Detection NA (1.49) vs Excitation NA (0.9)** — OTF uses detection NA; pattern freq uses excitation NA
4. **Bregman iteration structure** — `frac` must carry adjoint terms across iterations; resetting it kills the regularization
5. **Hessian $\lambda_B = 0.5$** — from `Bregman_Hessian_Denoise.m` (standalone version)

In [12]:
tifffile.imwrite('../results/hessian_sim.tif', hessian_result.astype(np.float32))
tifffile.imwrite('../results/tv_sim.tif', tv_result.astype(np.float32))
print('Results saved!')
print('\nOPTIMIZATION_FINISHED_SUCCESSFULLY')

Results saved!

OPTIMIZATION_FINISHED_SUCCESSFULLY
